# PrivateLocalAgent

**Kernel → Restart Kernel**，然后只运行下面 **一个** 代码单元格。

流程：加载真实智能体 → Doubao UI（含评委 10 问清单）→ 官方 `rc-tunnel` → 本页嵌入。  
云上代码偏旧时先 Terminal：`bash scripts/prep_and_run_notebook.sh`

In [ ]:
import os, sys, subprocess, threading
from pathlib import Path
from IPython.display import display, HTML, clear_output

def log(msg):
    print(msg, flush=True)

def progress(step, total, title):
    pct = int(100 * step / total)
    display(HTML(
        f"<div style='font-family:Segoe UI,PingFang SC,sans-serif;margin:6px 0;padding:10px 12px;"
        f"border-radius:10px;background:#0f172a;color:#e2e8f0;border:1px solid #334155'>"
        f"<b style='color:#5eead4'>[{step}/{total}] {title}</b>"
        f"<div style='margin-top:8px;height:6px;background:#1e293b;border-radius:999px'>"
        f"<div style='width:{pct}%;height:6px;background:#14b8a6;border-radius:999px'></div></div></div>"
    ))

try:
    ROOT = Path("/workspace/Radeon-hackathon-2026-07")
    if not (ROOT / "src" / "config.py").is_file():
        here = Path.cwd()
        ROOT = here.parent if here.name == "notebooks" else here
    os.chdir(ROOT)
    sys.path.insert(0, str(ROOT))
    log(f"ROOT={ROOT}")

    persist = Path("/workspace/persistence")
    if not persist.is_dir():
        persist = Path("/persistent")
    if persist.is_dir():
        os.environ.setdefault("PLA_DATA_ROOT", str(persist / "PrivateLocalAgent"))
        os.environ.setdefault("HF_HOME", str(persist / "huggingface"))
        Path(os.environ["PLA_DATA_ROOT"]).mkdir(parents=True, exist_ok=True)
        Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    os.environ.setdefault("USE_ROCM_AITER_ROPE_BACKEND", "0")
    os.environ["PLA_NOTEBOOK_UI_PORT"] = "7900"
    os.environ["PLA_ALLOW_PUBLIC"] = "1"
    os.environ["HTTP_HOST"] = "127.0.0.1"
    os.environ["HTTP_PORT"] = "7900"
    os.environ["PLA_OPEN_BROWSER"] = "0"
    os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ.get("PATH", "")

    progress(1, 4, "依赖检查（已装则跳过）")
    need = subprocess.run(
        [sys.executable, "-c", "import chromadb, transformers, PIL, rapidocr_onnxruntime"],
        capture_output=True,
    )
    if need.returncode != 0:
        log("[0] pip install…")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "chromadb", "sentence-transformers", "pypdf", "pyyaml", "python-dotenv",
            "pydantic", "openai", "transformers", "accelerate", "safetensors",
            "sentencepiece", "Pillow", "rapidocr-onnxruntime",
        ])
    else:
        log("[0] deps ok — skip pip")

    from src.agent.agent import PrivateAgent
    from src.agent.multi_agent import MultiAgentOrchestrator
    from src.agent.tools import ToolRegistry
    from src.apps.judge_script import ensure_judge_ocr_image
    from src.config import load_settings
    from src.llm.backend import build_llm
    from src.memory.memory import SessionMemory
    from src.privacy.audit import AuditTrail
    from src.rag.lazy_store import LazyVectorStore
    from src.skills import SkillRegistry
    from src.app.notebook_visual import launch_notebook_visual

    settings = load_settings()
    upload_dir = settings.resolve(settings.paths.upload_dir)
    ensure_judge_ocr_image(upload_dir)

    progress(2, 4, "准备工具与懒加载知识库")
    store = LazyVectorStore(settings)
    memory = SessionMemory(settings.resolve(settings.agent.memory_path))
    skills = SkillRegistry(settings.resolve(settings.paths.generated_projects))
    tools = ToolRegistry(store, memory, upload_dir, skill_registry=skills)
    audit = AuditTrail(settings.resolve("data/memory/audit.jsonl"))

    progress(3, 4, "加载本地 LLM（Radeon/ROCm）— 首次可能较久")
    llm = build_llm(settings.llm)
    agent = PrivateAgent(llm, tools, memory, settings.agent.max_steps, audit=audit)
    orch = MultiAgentOrchestrator(agent, tools)
    threading.Thread(target=store.warm, daemon=True).start()
    log("ready — launching UI (web + rc-tunnel)")

    progress(4, 4, "启动真实 Agent UI + rc-tunnel")
    ui = launch_notebook_visual(orch, default_mode="chat")
    log(f"local={ui.local_url}")
    log(f"public={ui.public_url}")
    log("左侧「评委清单」点题测试；可上传图片。")
except Exception as exc:
    display(HTML(
        "<div style='padding:14px;border-radius:12px;background:#450a0a;color:#fecaca;"
        "border:1px solid #f87171;font-family:Segoe UI,PingFang SC,sans-serif'>"
        "<b>启动失败</b><br/>请 Terminal 执行 "
        "<code>bash scripts/prep_and_run_notebook.sh</code>，"
        "然后 Kernel → Restart Kernel，再跑本单元格。"
        f"<pre style='white-space:pre-wrap;margin-top:10px'>{exc!r}</pre></div>"
    ))
    raise